# Brainclip Colab Voice Runtime — Fish S2-Pro

This notebook sets up an isolated virtual environment, installs Fish S2-Pro and pinned dependencies, starts a FastAPI voice runtime, and exposes it via Cloudflare Tunnel.

**Model:** `fishaudio/s2-pro` — 5B parameter dual-autoregressive TTS with inline emotion/prosody control via `[tag]` syntax. Zero-shot voice cloning from reference audio.

**GPU requirement:** T4 (16GB VRAM) or better. S2-Pro BF16 weights need ~10GB VRAM. Remaining headroom for KV cache and audio buffers.

## Why a virtual environment

Colab base packages change over time. We pin versions in a local venv so dependency drift does not silently break the voice server.

In [ ]:
from pathlib import Path
import os
import sys

APP_ROOT = Path('/content')
VENV_DIR = APP_ROOT / 'venv'
print('Python version:', sys.version)
if sys.version_info < (3, 10):
    raise RuntimeError('Brainclip runtime expects Python 3.10 or newer.')


In [ ]:
%%bash
set -e

# Install Node.js 18 & Chromium for Remotion
curl -fsSL https://deb.nodesource.com/setup_18.x | sudo -E bash -
sudo apt-get install -y nodejs chromium-browser ffmpeg

cd /content

# Clone / update repo
if [ ! -d "notebooks" ]; then
  git clone https://github.com/harshal0704/Brainclip.git temp_repo
  mv temp_repo/* .
  rm -rf temp_repo
fi

# Create venv and install pinned deps
python -m venv /content/venv
source /content/venv/bin/activate
python -m pip install --upgrade 'pip<25' 'setuptools<75' 'wheel<0.46'
python -m pip install -r /content/notebooks/requirements.txt

# Install remotion dependencies
cd /content/remotion && npm install


## Download Fish S2-Pro model (~9 GB)

This cell downloads the sharded S2-Pro checkpoint from HuggingFace. It only runs once per session — subsequent runs skip re-download.

**Estimated time:** 5–15 minutes on fast Colab internet.

In [ ]:
%%bash
set -e
source /content/venv/bin/activate

MODEL_DIR="/content/models/s2-pro"

# Skip if already fully downloaded
if [ -d "$MODEL_DIR" ] && [ "$(ls -1 $MODEL_DIR | wc -l)" -gt 5 ]; then
    echo "Model already present — skipping download."
    ls -lh "$MODEL_DIR"
else
    mkdir -p "$MODEL_DIR"
    huggingface-cli download fishaudio/s2-pro \
        --local-dir "$MODEL_DIR" \
        --local-dir-use-symlinks False
    echo "Download complete."
    ls -lh "$MODEL_DIR"
fi


## Verify GPU + VRAM

Check that CUDA is available and we have enough VRAM for S2-Pro (~10GB BF16).

In [ ]:
%%bash
set -e
source /content/venv/bin/activate
python -c "
import torch
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A')
if torch.cuda.is_available():
    mem_total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'VRAM total: {mem_total:.1f} GB')
    import torch.cuda
    mem_free, mem_allocated = torch.cuda.mem_get_info()
    print(f'VRAM free before load: {mem_free/1e9:.1f} GB')
    print(f'Recommended: 12+ GB free for S2-Pro.')
else:
    print('WARNING: No GPU detected. Fish S2-Pro requires CUDA.')
"


## Start FastAPI server + Cloudflare Tunnel

Run this cell to start the FastAPI voice runtime in the background, then start a Cloudflare Tunnel to expose it publicly. Copy the printed URL into Brainclip settings.

**Keep this cell running** — stopping it will terminate the server and tunnel.

In [ ]:
%%bash
set -e
source /content/venv/bin/activate
export BRAINCLIP_APP_ROOT=/content

# Kill any previous server on port 8000
pkill -f "uvicorn.*brainclip_colab_server" 2>/dev/null || true
sleep 1

# Start FastAPI server in background
nohup python -m uvicorn notebooks.brainclip_colab_server:app \
    --app-dir /content \
    --host 0.0.0.0 \
    --port 8000 \
    > /content/uvicorn.log 2>&1 &

sleep 8

# Verify server is up
curl -s http://127.0.0.1:8000/health | python -m json.tool || cat /content/uvicorn.log


In [ ]:
%%bash
set -e

# Install cloudflared if not present
which cloudflared >/dev/null 2>&1 || (
    python -m pip install cloudflared >/dev/null 2>&1 || true
    wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 \
        -O /usr/local/bin/cloudflared
    chmod +x /usr/local/bin/cloudflared
)

# Kill any existing tunnel
pkill -f "cloudflared tunnel" 2>/dev/null || true
sleep 2

# Start tunnel
cloudflared tunnel --url http://127.0.0.1:8000 \
    --logfile /content/cloudflared.log \
    --metrics 0.0.0.0:9100 \
    > /content/cloudflared.out 2>&1 &

sleep 8
TUNNEL_URL=$(grep -o 'https://[-a-zA-Z0-9.]*trycloudflare.com' /content/cloudflared.out | head -n 1)
echo ""
echo "=========================================="
echo "TUNNEL URL: $TUNNEL_URL"
echo "=========================================="
echo ""
echo "Copy the URL above into Brainclip → Settings → Colab Tunnel URL"
echo ""
# Verify tunnel is live
curl -s "${TUNNEL_URL}/health" | python -m json.tool || echo "Health check failed — wait 10s and try again"


## Operational notes

- Paste the Cloudflare URL above into Brainclip **Settings → Colab Tunnel URL**.
- The runtime fetches the latest scripts directly from the Brainclip repository.
- Models stay loaded in memory across jobs. Speaker token cache lives in `/content/cache/refs/`.
- GPU memory is shown in the `/health` endpoint. If it drops below 2 GB free, restart the runtime.
- If you see `model_oom` errors, restart the Colab runtime (Runtime → Disconnect and delete runtime → Run again).